In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split
import math
import torch.optim as optim
import itertools
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Global namefiles used for read csv files

In [3]:
global_path_train = 'data/final_train_data_transmission.csv'
global_path_test  = 'data/final_test_data_transmission.csv'
global_path_val   = 'data/final_val_data_transmission.csv'

# Positional Encoding

In [4]:
class TimePositionalEncoding(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        # Frecuencias fijas según Attention Is All You Need
        # Fixed frequencies according to Attention is all you need        
        # div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        i = torch.arange(0, d_model//2)
        div_term = torch.exp(-np.log(10000) * (2*i / d_model))
        self.register_buffer('div_term', div_term)

    def forward(self, delta_t):
        # batch x seq x d_model
        pe = torch.zeros(delta_t.size(0), delta_t.size(1), self.d_model, device=delta_t.device)
        # angle = (batch x seq x 1) x ( 1 x 1 x d_model//2)
        angle = delta_t.unsqueeze(-1) * self.div_term.unsqueeze(0).unsqueeze(0)
        pe[:, :, 0::2] = torch.sin(angle)
        pe[:, :, 1::2] = torch.cos(angle)
        return pe

# Model 

In [5]:
class LoraCollisionTransformer(nn.Module):        
    def __init__(self, num_numerical_features, d_model=64, n_heads=4, n_layers=2, dropout=0.1):
        # num_numerical_features- cantidad de variables fisicas (features)    
        # d_model               - Es la dimensión de representación interna del Transformer. Piensa en esto como la cantidad de "canales de información" que el modelo 
        #                         usará internamente para describir cada paquete. Todo el procesamiento profundo se hará en esta dimensión.
        # n_heads               - Le dice al Transformer en cuántas perspectivas diferentes debe dividir su atención.
        # n_layers              - Cuántas veces se repite el proceso de análisis (cuántos bloques Transformer se apilan).
        # dropout               - 
        super().__init__()
        self.num_num_features = num_numerical_features
        self.d_model = d_model
        
        # 1. Proyección directa de las variables físicas a d_model
        # Entran 'num_numerical_features' (ej. 6) y salen 'd_model' (ej. 64)
        # Recibe tu vector original de 6 variables físicas y lo transforma (lo proyecta) matemáticamente en un vector de tamaño d_model (64 dimensiones).
        # Recives the original input vector and project it to a vector size d_model
        self.input_projection = nn.Linear(num_numerical_features, d_model)
        
        # Apply TimePositionalEncoding 
        self.time_encoding = TimePositionalEncoding(d_model)
        
        # Apply encoder_layer
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, 
                                                   dim_feedforward=d_model*4, dropout=dropout, batch_first=True)        
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x shape: (Batch, Seq_Length, Features = 7) -> 6 físicas + 1 Delta_t
        
        # 1. DESEMPAQUETAR EL TENSOR X
        # Tomamos todas las columnas excepto la última (que es el tiempo)
        x_num = x[:, :, :self.num_num_features] 
        
        # La última columna es el Delta de Tiempo
        delta_t = x[:, :, -1] 
        
        # 2. PROYECCIÓN FÍSICA
        # Convertimos las 6 variables físicas en un vector de 64 dimensiones
        x_projected = self.input_projection(x_num)       
        
        # 3. POSITIONAL ENCODING (Inyección del tiempo)
        time_pe = self.time_encoding(delta_t)
        x_encoded = x_projected + time_pe
        
        # 4. TRANSFORMER
        transformer_out = self.transformer_encoder(x_encoded) 
        
        # 5. CLASIFICACIÓN
        target_packet = transformer_out[:, -1, :]             
        output = self.classifier(target_packet)               
        
        return output.squeeze(-1)

# Functions

In [10]:
def load_and_prepare_data(seq_length=16):    
    print("Loading dataset...")

    df_train = pd.read_csv(global_path_train)
    df_test  = pd.read_csv(global_path_test)
    df_val   = pd.read_csv(global_path_val)

    # Selecting continuous features
    continuous_features = ['latDev', 'longDev', 'elevSat', 'loraTP', 'loraSF', 'doppler', 'alt', 'raan']
    
    X_train_features = df_train[continuous_features].values
    #y_train          = df_train['rcvOk'].values

    X_test_features = df_test[continuous_features].values
    #y_test          = df_test['rcvOk'].values

    X_val_features = df_val[continuous_features].values


    scaler = StandardScaler()
    scaler.fit(np.concatenate((X_train_features, X_test_features, X_val_features), axis=0))

    #X_train = scaler.transform(X_train_features)
    #X_test  = scaler.transform(X_test_features)

    def create_sequences_data(df, scaler):
        cols_to_drop = ['dstId', 'srcSat', 'dstSat', 'loraCF', 'loraBW', 'loraCR', 'satId', 'srcId']
        df           = df.drop(columns=cols_to_drop)

        # Sort by id_simulaiton and time
        df = df.sort_values(by=['id_simulation', 'time']).reset_index(drop=True)
        
        # Apply StandardScaler         
        df[continuous_features] = scaler.transform(df[continuous_features])

        # Process data
        seq_X   = []
        label_y = []
        for sim_id, group_df in df.groupby('id_simulation'):
            num_features_array  = group_df[continuous_features].values
            ###cat_features_array  = group_df['srcId'].values
            time_array          = group_df['time'].values
            target_array        = group_df['rcvOk'].values
            num_packets         = len(group_df)
            
            # Discard if the simulation has less than SEQ_LENGHTS
            if num_packets < seq_length:
                continue
                        
            # Create sliding windows - Ventanas deslizantes
            for i in range(num_packets - seq_length + 1):
                # Select the window        
                window_num      = num_features_array[i : i + seq_length]
                ###window_cat      = cat_features_array[i : i + SEQ_LENGTH].reshape(-1, 1)
                window_times    = time_array[i : i + seq_length]
                window_targets  = target_array[i : i + seq_length]
                
                # El paquete central a predecir es el último de la ventana
                # The prediction should be the last elemento of the window
                target_idx      = seq_length - 1
                label_target    = window_targets[target_idx]
                time_target     = window_times[target_idx]
                            
                # Si un paquete ocurrió en el mismo segundo que el objetivo, delta_t = 0        
                delta_t = (window_times - time_target).reshape(-1, 1)
                print('window_times: ', window_times)
                print('time_target: ', time_target) 
                print('delta_t: ', delta_t)
                #delta_t =  time_target - window_times
                #print('delta_t: ', delta_t)
                
                #window_X = np.hstack((window_num, window_cat, delta_t))
                window_X = np.hstack((window_num, delta_t))

                seq_X.append(window_X)
                label_y.append(label_target)
                    
        X = torch.tensor(np.array(seq_X), dtype=torch.float32)
        y = torch.tensor(np.array(label_y), dtype=torch.float32)

        return X, y
    

    X_train, y_train = create_sequences_data(df_train, scaler)
    X_test , y_test  = create_sequences_data(df_test, scaler)
    X_val  , y_val   = create_sequences_data(df_val, scaler)


    return X_train, y_train, X_test, y_test, X_val, y_val

In [11]:
#def train_test(config, train_loader, val_loader, device, epochs=20, save_logs = False):    
def train_test(config, train_loader, val_loader, device, epochs=20):
    logs = {}
    logs['train_loss_log']  = []    
    logs['train_acc_log']   = []
    logs['val_loss_log']    = []
    logs['val_acc_log']     = []


    # Instanciar modelo con la configuración actual
    model = LoraCollisionTransformer(
        num_numerical_features  = 8, # 8 reales + 1 t_delta (el delta se adiciona despues)
        d_model                 = config['d_model'],
        n_heads                 = config['n_heads'],
        n_layers                = config['n_layers'],
        dropout                 = config['dropout']
    ).to(device)
    
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=config['lr'])
    
    #best_test_acc = 0.0   
    best_val_acc = 0.0 
    #best_val_loss = float('inf')
    avg_val_loss = 0.0
    for epoch in range(epochs):
        # Train
        model.train()
        train_loss      = 0.0
        correct_train   = 0
        total_train     = 0

        #for X_b, y_b in train_loader:
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            # Forward pass
            predictions = model(batch_X) # Usa internamente el squeeze(-1)

            # Clean gradient 
            optimizer.zero_grad()

            # Calculate error loss
            loss = criterion(predictions, batch_y)

            # Backpropagation
            loss.backward()

            # Update weights
            optimizer.step()

            # loss.item - The average of the batch losses will give you an estimate of the “epoch loss” during training. Returns the value of this tensor as a standard Python number 
            train_loss += loss.item() * batch_X.size(0)
            
            # Convert to 0 - 1 the vector batch
            predictions_class = (predictions >= 0.5).float()
            
            correct_train += (predictions_class == batch_y).sum().item()
            total_train   += batch_y.size(0)

            
        
        avg_train_loss  = train_loss / total_train
        train_acc       = correct_train / total_train
        
        logs['train_loss_log'].append(avg_train_loss)
        logs['train_acc_log'].append(train_acc)

        # Validation
        model.eval()        
        val_loss     = 0.0  
        correct_val = 0
        total_val   = 0
        
        y_true = []
        y_pred = []
        with torch.no_grad():
            #for X_b, y_b in val_loader:
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                
                # Predict
                predictions = model(batch_X)
                loss        = criterion(predictions, batch_y)
                val_loss   += loss.item() * batch_X.size(0)

                # Calculate correct or not 
                predictions_class = (predictions >= 0.5).float()
                correct_val      += (predictions_class == batch_y).sum().item()
                total_val        += batch_y.size(0)

                # Other method to calculate metrics
                y_pred.extend(predictions_class.cpu().numpy())  
                y_true.extend(batch_y.cpu().numpy())
                
        
        avg_val_loss = val_loss / total_val
        val_acc      = correct_val / total_val

        
        logs['val_loss_log'].append(avg_val_loss)
        logs['val_acc_log'].append(val_acc)
        
        # Print metrics
        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        auc = roc_auc_score(y_true, y_pred)
        

        print(f"Epoch {epoch+1:02d}/{epochs} | "
        f"Train Loss: {avg_train_loss:.4f} - Acc: {train_acc*100:.1f}% | "
        f"Val Loss: {avg_val_loss:.4f} - Acc: {val_acc*100:.1f}% | " 
        f"Val Acc: {acc:.4f} - Val Prec: {prec*100:.1f}% | "
        f"Val Rec: {rec:.4f} - Val F1: {f1*100:.1f}% | "
        f"Val AUC-ROC: {auc:.4f} "
        )

    
    return avg_val_loss, model, logs

In [12]:
def fine_tuning(train_dataset, test_dataset, val_dataset):
    device = "cpu"
    print(f"Using : {device}")    
    #device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    
    # Setting params
    
    param_grid = {
        'd_model'   : [32, 64], #128                # Dimension of the model
        'n_heads'   : [2, 4],                       # Number of projections
        'n_layers'  : [1, 2, 3],                    # Number of encoder layers
        'dropout'   : [0.1, 0.2], # , 0.2                # doprout
        'lr'        : [0.001], #, 0.0005            # Learning rate
        'batch_size': [32]                          # Batch Size
    }
    '''
    param_grid = {
        'd_model'   : [32, ], #128                # Dimension of the model
        'n_heads'   : [2, ],                       # Number of projections
        'n_layers'  : [1, ],                    # Number of encoder layers
        'dropout'   : [0.1,], # , 0.2                # doprout
        'lr'        : [0.001], #, 0.0005            # Learning rate
        'batch_size': [32]                          # Batch Size
    }
    '''
    # Generate the combination of parameters
    keys, values = zip(*param_grid.items())
    combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
    
    print(f"Starting  {len(combinations)} combinations ...")
    print("="*60)
    
    #print(combinations)

    results     = []    
    start_total = time.time()  
    best_val_loss = float('inf')  
    best_logs = None
    name_model = ''
    for i, config in enumerate(combinations):
        # Create DataLoaders
        train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
        test_loader  = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False)
        val_loader   = DataLoader(val_dataset, batch_size=config['batch_size'], shuffle=False)
        
        print(f"[{i+1}/{len(combinations)}] Testing: {config} ...", end=" ")
                
        # Training
        final_loss, model, logs = train_test(config, train_loader, val_loader, device, epochs=20)
        
        print(f"-> Best Loss: {final_loss:.4f}")
            
        if final_loss < best_val_loss:
            print(f"New best model found with val_loss: {final_loss:.4f} (previous best: {best_val_loss:.4f}). Saving model...")
            name_model = 'Transformer_' + str(config['d_model']) + '_' + str(config['n_heads']) + '_loss_' + str(round(final_loss, 4)) + '.pth'
            best_val_loss = final_loss
            best_logs = logs
            torch.save(model.state_dict(), name_model)
            print("Model saved !!!")


        # Guardar resultado
        res             = config.copy()
        res['loss'] = final_loss
        results.append(res)
        #'''
    
    print("="*60)
    print(f"Finishing in {(time.time() - start_total)/60:.1f} minutes.")
        
    #results_df = pd.DataFrame(results)
    #results_df = results_df.sort_values(by='accuracy', ascending=False)
    
    #print("Best results")
    #print(results_df.head(5))
    
    # Save to CSV
    #results_df.to_csv('hyperparameter_results.csv', index=False)
    
    
    # Return best configuration
    best_config = config#.to_dict() #results_df.iloc[0].to_dict()
    return best_config, name_model, best_logs

In [8]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [13]:
X_train, y_train, X_test, y_test, X_val  , y_val = load_and_prepare_data(seq_length = 16)
train_dataset = TensorDataset(X_train, y_train)
test_dataset  = TensorDataset(X_test, y_test)
val_dataset   = TensorDataset(X_val, y_val)

Loading dataset...
window_times:  [11.1269 12.3645 13.0547 13.3005 15.3593 18.1189 18.4697 18.6216 22.206
 22.8768 23.4428 24.0932 26.6197 29.4238 31.6479 34.208 ]
time_target:  34.208
delta_t:  [[-23.0811]
 [-21.8435]
 [-21.1533]
 [-20.9075]
 [-18.8487]
 [-16.0891]
 [-15.7383]
 [-15.5864]
 [-12.002 ]
 [-11.3312]
 [-10.7652]
 [-10.1148]
 [ -7.5883]
 [ -4.7842]
 [ -2.5601]
 [  0.    ]]
window_times:  [12.3645 13.0547 13.3005 15.3593 18.1189 18.4697 18.6216 22.206  22.8768
 23.4428 24.0932 26.6197 29.4238 31.6479 34.208  37.2052]
time_target:  37.2052
delta_t:  [[-24.8407]
 [-24.1505]
 [-23.9047]
 [-21.8459]
 [-19.0863]
 [-18.7355]
 [-18.5836]
 [-14.9992]
 [-14.3284]
 [-13.7624]
 [-13.112 ]
 [-10.5855]
 [ -7.7814]
 [ -5.5573]
 [ -2.9972]
 [  0.    ]]
window_times:  [13.0547 13.3005 15.3593 18.1189 18.4697 18.6216 22.206  22.8768 23.4428
 24.0932 26.6197 29.4238 31.6479 34.208  37.2052 40.5622]
time_target:  40.5622
delta_t:  [[-27.5075]
 [-27.2617]
 [-25.2029]
 [-22.4433]
 [-22.0925]
 [-

/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


[462.234 463.734 463.775 466.072 466.939 476.697 477.387 478.665 480.048
 480.58  481.419 481.855 483.483 484.478 486.353 486.707]
time_target:  486.707
delta_t:  [[-24.473]
 [-22.973]
 [-22.932]
 [-20.635]
 [-19.768]
 [-10.01 ]
 [ -9.32 ]
 [ -8.042]
 [ -6.659]
 [ -6.127]
 [ -5.288]
 [ -4.852]
 [ -3.224]
 [ -2.229]
 [ -0.354]
 [  0.   ]]
window_times:  [463.734 463.775 466.072 466.939 476.697 477.387 478.665 480.048 480.58
 481.419 481.855 483.483 484.478 486.353 486.707 487.077]
time_target:  487.077
delta_t:  [[-23.343]
 [-23.302]
 [-21.005]
 [-20.138]
 [-10.38 ]
 [ -9.69 ]
 [ -8.412]
 [ -7.029]
 [ -6.497]
 [ -5.658]
 [ -5.222]
 [ -3.594]
 [ -2.599]
 [ -0.724]
 [ -0.37 ]
 [  0.   ]]
window_times:  [463.775 466.072 466.939 476.697 477.387 478.665 480.048 480.58  481.419
 481.855 483.483 484.478 486.353 486.707 487.077 487.84 ]
time_target:  487.84
delta_t:  [[-24.065]
 [-21.768]
 [-20.901]
 [-11.143]
 [-10.453]
 [ -9.175]
 [ -7.792]
 [ -7.26 ]
 [ -6.421]
 [ -5.985]
 [ -4.357]
 [ -3.36

/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/ylnner/miniconda3/envs/p_new/lib/python3.14/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


[[-22.106]
 [-16.619]
 [-16.4  ]
 [-13.498]
 [-11.368]
 [-11.272]
 [ -8.627]
 [ -7.981]
 [ -7.859]
 [ -7.513]
 [ -7.376]
 [ -5.012]
 [ -4.424]
 [ -4.09 ]
 [ -1.32 ]
 [  0.   ]]
window_times:  [ 982.878  983.097  985.999  988.129  988.225  990.87   991.516  991.638
  991.984  992.121  994.485  995.073  995.407  998.177  999.497 1001.31 ]
time_target:  1001.31
delta_t:  [[-18.432]
 [-18.213]
 [-15.311]
 [-13.181]
 [-13.085]
 [-10.44 ]
 [ -9.794]
 [ -9.672]
 [ -9.326]
 [ -9.189]
 [ -6.825]
 [ -6.237]
 [ -5.903]
 [ -3.133]
 [ -1.813]
 [  0.   ]]
window_times:  [ 983.097  985.999  988.129  988.225  990.87   991.516  991.638  991.984
  992.121  994.485  995.073  995.407  998.177  999.497 1001.31  1002.04 ]
time_target:  1002.04
delta_t:  [[-18.943]
 [-16.041]
 [-13.911]
 [-13.815]
 [-11.17 ]
 [-10.524]
 [-10.402]
 [-10.056]
 [ -9.919]
 [ -7.555]
 [ -6.967]
 [ -6.633]
 [ -3.863]
 [ -2.543]
 [ -0.73 ]
 [  0.   ]]
window_times:  [ 985.999  988.129  988.225  990.87   991.516  991.638  991.984  9

In [ ]:
X_train.shape

In [10]:
best_params, best_name_model, best_logs = fine_tuning(train_dataset, test_dataset, val_dataset)
print("Best configuration : ")
print(best_params)
print(f"Best model name: {best_name_model}")

Using : cpu
Starting  24 combinations ...
[1/24] Testing: {'d_model': 32, 'n_heads': 2, 'n_layers': 1, 'dropout': 0.1, 'lr': 0.001, 'batch_size': 32} ... Epoch 01/20 | Train Loss: 0.4307 - Acc: 80.3% | Val Loss: 0.2501 - Acc: 89.9% | Val Acc: 0.8995 - Val Prec: 82.9% | Val Rec: 0.9100 - Val F1: 86.8% | Val AUC-ROC: 0.9017 
Epoch 02/20 | Train Loss: 0.3340 - Acc: 84.2% | Val Loss: 0.2071 - Acc: 90.9% | Val Acc: 0.9085 - Val Prec: 83.1% | Val Rec: 0.9375 - Val F1: 88.1% | Val AUC-ROC: 0.9148 
Epoch 03/20 | Train Loss: 0.3058 - Acc: 85.4% | Val Loss: 0.2046 - Acc: 91.0% | Val Acc: 0.9103 - Val Prec: 86.8% | Val Rec: 0.8875 - Val F1: 87.8% | Val AUC-ROC: 0.9054 
Epoch 04/20 | Train Loss: 0.2984 - Acc: 86.1% | Val Loss: 0.1901 - Acc: 91.8% | Val Acc: 0.9176 - Val Prec: 87.2% | Val Rec: 0.9050 - Val F1: 88.8% | Val AUC-ROC: 0.9149 
Epoch 05/20 | Train Loss: 0.2834 - Acc: 86.8% | Val Loss: 0.1963 - Acc: 91.5% | Val Acc: 0.9149 - Val Prec: 85.4% | Val Rec: 0.9225 - Val F1: 88.7% | Val AUC-ROC:

In [ ]:
best_params = {'d_model': 32, 'n_heads': 4, 'n_layers': 3, 'dropout': 0.1, 'lr': 0.001, 'batch_size': 32}
#best_name_model = 'Transformer_32_2_loss_0.0737.pth'


#best_params = {'d_model': 64, 'n_heads': 4, 'n_layers': 3, 'dropout': 0.2, 'lr': 0.001, 'batch_size': 32}
#best_name_model = 'Transformer_64_4_loss_0.1041.pth'


{'d_model': 32, 'n_heads': 4, 'n_layers': 3, 'dropout': 0.1, 'lr': 0.001, 'batch_size': 32}
 {'d_model': 32, 'n_heads': 2, 'n_layers': 3, 'dropout': 0.1, 'lr': 0.001, 'batch_size': 32}
-> Best Loss: 0.0708
New best model found with val_loss: 0.0708 (previous best: 0.0958). Saving model...

In [11]:
best_params =  {'d_model': 32, 'n_heads': 2, 'n_layers': 3, 'dropout': 0.1, 'lr': 0.001, 'batch_size': 32}
best_params

{'d_model': 32,
 'n_heads': 2,
 'n_layers': 3,
 'dropout': 0.1,
 'lr': 0.001,
 'batch_size': 32}

In [12]:
best_name_model

'Transformer_32_2_loss_0.0708.pth'

In [13]:
print("\n--- EVALUACIÓN FINAL EN TEST SET ---")
device = "cpu"
# Instanciar modelo con la configuración actual
final_model = LoraCollisionTransformer(
    num_numerical_features  = 8, # 8 reales + 1 t_delta (el delta se adiciona despues)
    d_model                 = best_params['d_model'],
    n_heads                 = best_params['n_heads'],
    n_layers                = best_params['n_layers'],
    dropout                 = best_params['dropout']
).to(device)

#name_to_load = 'Transformer_32_4_loss_0.0882.pth'


test_loader  = DataLoader(test_dataset, batch_size=best_params['batch_size'], shuffle=False)

# 2. Cargas los pesos guardados (.pth)
final_model.load_state_dict(torch.load(best_name_model))
final_model.eval() # Modo evaluación 

criterion = nn.BCELoss()

#optimizer = optim.Adam(model_lstm.parameters(), lr=0.001)

# 3. Evaluación final (Una sola pasada)
test_loss = 0.0
correct_test = 0
total_test = 0
y_true, y_pred = [], []
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        # Solo predecimos y calculamos error, NO hacemos .backward() ni .step()
        predictions = final_model(batch_X)
        loss = criterion(predictions, batch_y)
        
        # Estadísticas
        test_loss += loss.item() * batch_X.size(0)
        preds_clase = (predictions >= 0.5).float()
        correct_test += (preds_clase == batch_y).sum().item()
        total_test += batch_y.size(0)

        # Other method to calculate metrics
        y_pred.extend(preds_clase.cpu().numpy())  
        y_true.extend(batch_y.cpu().numpy())


 # --- REPORTE DEL EPOCH ---
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
auc = roc_auc_score(y_test, y_pred)
print(f"FINAL REPORT"        
        f"Test Acc: {acc:.4f} - Test Prec: {prec*100:.1f}% | "
        f"Test Rec: {rec:.4f} - Test F1: {f1*100:.1f}% | "
        f"Test AUC-ROC: {auc:.4f}"
        )


print(f"F1 Final en Test: {f1_score(y_true, y_pred):.4f}")


--- EVALUACIÓN FINAL EN TEST SET ---
FINAL REPORTTest Acc: 0.9257 - Test Prec: 93.0% | Test Rec: 0.9084 - Test F1: 91.9% | Test AUC-ROC: 0.9246
F1 Final en Test: 0.9191


In [ ]:
FINAL REPORTTest Acc: 0.9312 - Test Prec: 92.6% | Test Rec: 0.9449 - Test F1: 93.5% | Test AUC-ROC: 0.9304
F1 Final en Test: 0.9353

In [ ]:
scripted = torch.jit.script(final_model)
scripted.save("model_scripted.pt")

In [ ]:
type(final_model)

In [ ]:
total_params = sum(p.numel() for p in final_model.parameters())
print(f"Total parameters: {total_params}")

# Load model

In [ ]:
train_csv.head()

In [ ]:

model = torch.jit.load("model_scripted.pt")
model.eval()

# --- A. Print full model architecture ---
#print(model)
for name, param in model.named_parameters():
    print(f"{name}: {param.shape}")
    break  # first layer is enough

In [ ]:
for nombre, parametro in model.named_parameters():
    if 'weight' in nombre:
        print(f"Primera capa: '{nombre}'")
        print(f"Dimensiones de los pesos: {parametro.shape}")
        # La forma suele ser [neuronas_salida, features_de_entrada]
        print(f"-> Tu modelo espera exactamente {parametro.shape[1]} features de entrada.")
        break # Detenemos el bucle porque solo nos importa la primera capa

In [ ]:
dummy = torch.zeros(1,  8)

dummy

In [ ]:
# --- C. Confirm with a dummy forward pass ---
dummy = torch.randn(1, 16, 8)   # batch=1, features=3
#dummy = torch.zeros(1, 8)
output = model(dummy)
print("Output shape:", output.shape)  # torch.Size([1, 2]) → 2 classes

In [ ]:
output

In [ ]:
def simple_execution(config, test_loader):  
    #X, y = load_and_prepare_data(seq_length = 16)
    #X_train, y_train, X_test, y_test, X_val  , y_val = load_and_prepare_data(seq_length = 16)
    #train_dataset = TensorDataset(X_train, y_train)
    #test_dataset  = TensorDataset(X_test, y_test)
    #val_dataset   = TensorDataset(X_val, y_val)
    
    #device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    device = "cpu"
    print(f"Using : {device}")
        
    # Train and test with best params
    #train_loader = DataLoader(train_dataset, batch_size=int(config['batch_size']), shuffle=True)
    #test_loader  = DataLoader(test_dataset, batch_size=int(config['batch_size']), shuffle=False)
    #val_loader   = DataLoader(val_dataset, batch_size=int(config['batch_size']), shuffle=False)

    final_acc, logs = train_test(config, train_loader, test_loader, device, epochs=20, save_logs=True)
    print(f"Best Test Accuracy: {final_acc*100:.2f}%")

    #train_loss = logs['']
    #test_loss  = logs['']
    print("Final logs: " , logs)
    #print("Final Train loss: " , test_loss)
    return logs


In [ ]:
#type(best_params['d_model'])
best_params

In [ ]:
best_params = {'d_model': 64.0,
 'n_heads': 2.0,
 'n_layers': 2.0,
 'dropout': 0.1,
 'lr': 0.001,
 'batch_size': 32.0,
 'accuracy': 0.952076677316294}

In [ ]:
int_fields = ["d_model", "n_heads", "n_layers", "batch_size"]

for key in int_fields:
    best_params[key] = int(best_params[key])


logs = simple_execution(best_params)

In [ ]:
logs.keys()

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, 21)

plt.figure(figsize=(14, 6))

# ACCURACY
plt.subplot(1, 2, 1)
plt.plot(epochs_range, logs['train_acc_log'], label='Training', marker='o')
plt.plot(epochs_range, logs['test_acc_log'], label='Testing', marker='o', linestyle='--')
plt.title('Accuracy evolution over epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True)

# LOSS
plt.subplot(1, 2, 2)
plt.plot(epochs_range, logs['train_loss_log'], label='Training', marker='o')
plt.plot(epochs_range, logs['test_loss_log'], label='Testing', marker='o', linestyle='--')
plt.title('Loss over epochs')
plt.xlabel('Epochs')
plt.ylabel('Binary Cross Entropy Loss')
plt.legend(loc='upper right')
plt.grid(True)


plt.suptitle('Acc and Loss for the best config')
plt.tight_layout()
plt.show()

In [ ]:
!conda install matplotlib

In [ ]:
y